# Learn-a-World Phase 2: Dual-Store Online Causal Memory Screening (Arm C)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sahiltolani30/learn-a-world/blob/main/notebooks/06_screen_dual_store_colab.ipynb)

This notebook executes the **50-episode official screening (seeds 2050–2099)** on GPU for **Arm C: Dual-Store Online Causal Memory**.

### Key Architecture Highlights:
- **Neocortical Semantic Core + Quarantined Sandbox**: Hypotheses are isolated until verified.
- **Physical Invariance Gate**: Zero spurious hypotheses from sensor noise.
- **Bayesian Falsification Pruning**: Non-productive hypotheses are evicted; rare milestones are protected.
- **Active-Link Topological Gating**: Eliminates premature execution at crafting stations.
- **Dual Memory Serialization**: Native support for both `.json` and PyTorch `.pt` memory bank formats.
- **Zero-Hardcoding Guarantee**: 100% compliant with the strict sensory protocol (RGB + scalar reward only).


### Step 1: Mount Google Drive
Mounts Google Drive to load checkpoints and mirror screening results.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/learn-a-world")
(DRIVE_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / "results").mkdir(parents=True, exist_ok=True)

print("Google Drive workspace ready at:", DRIVE_DIR)

### Step 2: Clone or Pull Repository on Branch `main`

In [ ]:
%cd /content
import os
if not os.path.exists("learn-a-world"):
    !git clone https://github.com/sahiltolani30/learn-a-world.git
%cd /content/learn-a-world
!git fetch origin
!git checkout main
!git pull origin main
!git log -1 --oneline


### Step 3: Install Dependencies & Check GPU

In [ ]:
%cd /content/learn-a-world
!pip install -q -r requirements.txt
!pip install -q crafter gymnasium pytest
!nvidia-smi

### Step 4: Run Pre-Flight Automated Unit & Anti-Hardcoding Tests

In [ ]:
%cd /content/learn-a-world
!python3 scripts/strict_anti_hardcoding_audit.py
!pytest tests/test_phase2_*.py tests/test_dual_store_backward_compatibility.py -v

### Step 5: Verify & Copy Checkpoint and Dual-Store Memory Banks

In [ ]:
import shutil
from pathlib import Path

# 1. Model Checkpoint (Auto-detects jepa_crafter_v3_state_readout_v2.pt)
drive_dir = Path("/content/drive/MyDrive/learn-a-world")
ckpt_names = ["jepa_crafter_v3_state_readout_v2.pt", "jepa_crafter_v3_phase2.pt"]
active_ckpt = None
for cname in ckpt_names:
    drive_ckpt = drive_dir / "checkpoints" / cname
    local_ckpt = Path(f"/content/learn-a-world/checkpoints/{cname}")
    local_ckpt.parent.mkdir(parents=True, exist_ok=True)
    if drive_ckpt.exists() and not local_ckpt.exists():
        shutil.copy(drive_ckpt, local_ckpt)
        print(f"Copied {drive_ckpt} -> {local_ckpt}")
    if local_ckpt.exists() and active_ckpt is None:
        active_ckpt = local_ckpt
        print(f"Active model checkpoint: {local_ckpt} ({local_ckpt.stat().st_size / 1e6:.1f} MB)")

# 2. Dual-Store Base Memory Bank (.json and .pt)
drive_bank_json = drive_dir / "checkpoints/dual_store_causal_memory_bank.json"
drive_bank_pt = drive_dir / "checkpoints/dual_store_memory_bank.pt"
local_bank_json = Path("/content/learn-a-world/checkpoints/dual_store_causal_memory_bank.json")
local_bank_pt = Path("/content/learn-a-world/checkpoints/dual_store_memory_bank.pt")

if drive_bank_json.exists() and not local_bank_json.exists():
    shutil.copy(drive_bank_json, local_bank_json)
    print(f"Loaded Dual-Store JSON bank from Drive: {drive_bank_json}")
if drive_bank_pt.exists() and not local_bank_pt.exists():
    shutil.copy(drive_bank_pt, local_bank_pt)
    print(f"Loaded Dual-Store PT bank from Drive: {drive_bank_pt}")

if local_bank_json.exists():
    print(f"Dual-Store bank ready: {local_bank_json} ({local_bank_json.stat().st_size / 1e6:.1f} MB)")
else:
    print("Dual-Store bank will be consolidated from offline shards in Step 5b.")


### Step 5b: (Optional) Consolidate / Train Offline Dual-Store Memory (.json & .pt)
Run this cell to consolidate the offline memory bank from trajectory shards using `jepa_crafter_v3_state_readout_v2.pt`. It produces both `dual_store_causal_memory_bank.json` and `dual_store_memory_bank.pt` with clean causal needs, no spurious vitals edges, and auto-mirrors them to Google Drive.

In [ ]:
%cd /content/learn-a-world
!python3 dual_store/consolidate_memory.py \
  --checkpoint checkpoints/jepa_crafter_v3_state_readout_v2.pt \
  --output_json checkpoints/dual_store_causal_memory_bank.json \
  --output_pt checkpoints/dual_store_memory_bank.pt \
  --max_episodes 300 \
  --device cuda \
  --drive_dir /content/drive/MyDrive/learn-a-world

!python3 scripts/patch_bank_pickaxe.py


### Step 6: Launch 50-Episode GPU Screening (Seeds 2050–2099)
Executes Recurrent MPC with online dual-store learning on GPU, using our own offline memory bank and persisting lessons to `results/dual_store_memory_bank_online.json`.

In [ ]:
%cd /content/learn-a-world
!python3 scripts/screen_dual_store_50eps.py \
  --checkpoint checkpoints/jepa_crafter_v3_state_readout_v2.pt \
  --causal_memory_bank checkpoints/dual_store_causal_memory_bank.json \
  --online_learning \
  --online_bank_path results/dual_store_memory_bank_online.json \
  --output_json results/screen_v3_armC_dual_store_50ep.json \
  --use_transition_model \
  --transition_scoring \
  --death_risk_factor 0.5 \
  --horizon 10 \
  --num_candidates 64 \
  --cem_iterations 2 \
  --device cuda \
  --drive_dir /content/drive/MyDrive/learn-a-world


### Step 7: View Summary & Compare Against Baseline

In [ ]:
import json
from pathlib import Path

res_file = Path("/content/learn-a-world/results/screen_v3_armC_dual_store_50ep.json")
if not res_file.exists():
    res_file = Path("/content/drive/MyDrive/learn-a-world/results/screen_v3_armC_dual_store_50ep.json")

if res_file.exists():
    with open(res_file) as f:
        data = json.load(f)
    print(f"Official Crafter Score: {data.get('crafter_score', 0):.2f}%")
    print(f"Mean Episode Reward: {data.get('mean_episode_reward', 0):.2f}")
    print(f"Mean Survival Length: {data.get('mean_survival_length', 0):.1f}")
    print("\nUnlock Percentages:")
    for k, v in data.get("unlock_percentages", {}).items():
        if v > 0:
            print(f"  {k:22s}: {v:.1f}%")
else:
    print("Results file not found.")


### Step 8: Inspect In-Game Decisions & Memory Updates (Forensic Debugger)
Run this cell after screening to see:
1. Exact step-by-step subgoal decision traces (how the planner chose to harvest, craft table, or craft pickaxe).
2. Which online memory cards were created, confirmed, or denied cross-episode.
3. Full markdown report in .

In [ ]:
%cd /content/learn-a-world
# 1. Inspect in-game decisions and subgoal traces
!python3 scripts/inspect_episode_decisions.py --results results/screen_v3_armC_dual_store_50ep.json

# 2. Inspect newly learned online memory cards
!python3 scripts/export_online_memory_review.py results/dual_store_memory_bank_online.json
